In [7]:
import pandas as pd
import json
import requests
import time

In [3]:
data = pd.read_excel('batch_v1_size_1000.xlsx')
data.head(3)

,Unnamed: 0,id,hh_vac_id,hh_vac_link,title,experience,employment,schedule,salary_raw,salary_from,salary_to,currency,employer,address_raw,area,skills,published_at,raw_page
0,0,1,131963477,https://hh.ru/vacancy/131963477,Senior AI Multiagent Systems Engineer,Опыт более 6 лет,NaN,NaN,"400 000 – 700 000₽за месяц,на руки",NaN,NaN,NaN,Simplenight,Москва,NaN,NaN,NaN,"<!DOCTYPE html>\n<html class=""desktop"" lang=""r..."
1,1,2,132530194,https://hh.ru/vacancy/132530194,Middle+ AI-инженер,Опыт 1-3 года,NaN,NaN,"до320 000₽за месяц,до вычета налогов",NaN,NaN,NaN,Точка Банк,Москва,NaN,[],2026-04-27T18:53:55.340855,NaN
2,2,3,131744414,https://hh.ru/vacancy/131744414,Prompt Engineer / AI‑редактор (нейросети и авт...,Опыт 1-3 года,полная,полный день,"80 000 – 100 000₽за месяц,на руки",80000.0,100000.0,RUR,ДЖЕЙКЕТ РАБОТА,Москва,Москва,[],2026-04-27T18:54:50.686408,"<!DOCTYPE html>\n<html class=""desktop"" lang=""r..."


In [4]:
PROMPT = """
Ты — система извлечения структурированных данных из вакансий.

ЗАДАЧА:
Проанализируй вакансию и извлеки:
1. skills — профессиональные навыки кандидата
2. level — уровень позиции
3. category — категория профессии

ИСПОЛЬЗУЙ ТОЛЬКО ДАННЫЕ ИЗ ВАКАНСИИ:
- title
- experience
- employment
- schedule
- salary_raw
- employer
- area
- description / raw_page_text

НЕЛЬЗЯ выдумывать требования, которых нет в тексте.

==================================================
1. ПОЛЕ skills
==================================================

Извлекай профессиональные навыки, инструменты, знания, компетенции, необходимые для работы.

Допускаются НЕ ТОЛЬКО IT skills.

-----------------------------------
Разрешённые типы навыков:
-----------------------------------

A. Технические / IT:
Python, SQL, Excel, Docker, AutoCAD, 1C, Figma, JavaScript

B. Финансы / офис:
Бухгалтерский учет
МСФО
1С Бухгалтерия
Налоговый учет
Финансовый анализ
Бюджетирование

C. Продажи / маркетинг:
B2B продажи
CRM
Холодные продажи
Контекстная реклама
SEO
SMM
Email-маркетинг
Переговоры

D. Производство / логистика:
Складской учет
Управление запасами
Закупки
Логистика
SAP
Контроль качества

E. HR / рекрутинг:
Подбор персонала
Interviewing
Boolean Search
Адаптация персонала
Кадровое делопроизводство

F. Медицина / фарма:
Сестринское дело
УЗИ
Фармакология
Работа с мед. документацией

G. Юриспруденция:
Договорная работа
Корпоративное право
Судебная практика
Комплаенс

-----------------------------------
Что НЕЛЬЗЯ включать:
-----------------------------------

стрессоустойчивость  
ответственность  
коммуникабельность  
обучаемость  
пунктуальность  
работа в команде  
нацеленность на результат  

Это soft skills — их не добавлять.

-----------------------------------
Правила:
-----------------------------------

1. Только подтверждённые вакансией навыки.
2. Нормализуй названия навыков.
3. Без дублей.
4. Максимум 15 skills.
5. Если навыков нет явно — можно вывести из названия вакансии.
6. Если данных недостаточно — [].

==================================================
2. ПОЛЕ level
==================================================

Одно значение:

junior
middle
senior
lead
head
unknown

Правила:

- стажер / без опыта / intern => junior
- 1-3 года => junior или middle
- 3-6 лет => middle или senior
- 6+ лет => senior или lead
- руководитель отдела => head
- lead => lead

==================================================
3. ПОЛЕ category
==================================================

Одно значение из списка:

AI Engineer
Software Developer
Data Scientist
ML Engineer
Data Engineer
Accountant
Financial Analyst
Sales Manager
Marketing Manager
HR Specialist
Recruiter
Lawyer
Doctor
Nurse
Designer
Project Manager
Product Manager
Operations Manager
Logistics Specialist
Teacher
Customer Support
Administrator
Engineer
Other

Определи по смыслу вакансии.

==================================================
ФОРМАТ ОТВЕТА
==================================================

Верни СТРОГО JSON.
Без markdown.
Без комментариев.
Без текста до или после.

{
  "skills": ["Excel", "Бухгалтерский учет", "1С Бухгалтерия"],
  "level": "middle",
  "category": "Accountant"
}

==================================================
ЕСЛИ НЕТ ДАННЫХ:
==================================================

{
  "skills": [],
  "level": "unknown",
  "category": "Other"
}
"""

In [ ]:
API_KEY = "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"
MODEL = "openai/gpt-4o-mini"

def enrich_vacancy(row):

    prompt = f"""
    Вакансия:
    title: {row['title']}
    experience: {row['experience']}
    salary: {row['salary_raw']}
    employer: {row['employer']}
    area: {row['area']}
    text: {str(row['raw_page'])[:4000]}
    """

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": PROMPT},
            {"role": "user", "content": prompt}
        ],
        "response_format": {"type": "json_object"}
    }

    r = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=data
    )

    txt = r.json()["choices"][0]["message"]["content"]

    try:
        return json.loads(txt)
    except:
        return {"skills": [], "level": None, "category": None}

In [ ]:
results = []

for _, row in data.iterrows():
    print(f'Начало : {_}')

    res = enrich_vacancy(row)
    results.append(res)
    time.sleep(1)

    print(f'Конец : {_}')


extra = pd.DataFrame(results)
extra.to_excel('extra.xlsx')

df = pd.concat([data, extra], axis=1)
df.to_excel('df.xlsx')